In [13]:
import torch
import torch.nn as nn
from torch.optim import SGD
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision 
import numpy as np
import matplotlib.pyplot as plt 
from torchvision.datasets import MNIST
from torchvision import transforms

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [14]:
from torchvision.datasets import MNIST
from torchvision import transforms

train_dataset = MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

In [15]:
train = MNIST("data", train=True, download=False, transform=transforms.ToTensor())
test = MNIST("data", train=False, download=False, transform=transforms.ToTensor())

In [16]:
from torch.utils.data import Dataset

class MNISTDataset(Dataset):

    def __init__(self, mnist_dataset):
        self.dataset = mnist_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return image, label

In [17]:
L = nn.CrossEntropyLoss()

In [18]:
class MyNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.Matrix1 = nn.Linear(28**2, 100)
        self.Matrix2 = nn.Linear(100, 50)
        self.Matrix3 = nn.Linear(50, 10)
        self.R = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28**2)
        x = self.R(self.Matrix1(x))
        x = self.R(self.Matrix2(x))
        x = self.Matrix3(x)
        return x

In [19]:
f = MyNeuralNet().to(device)

In [20]:
def train_model(dl, f, n_epochs=20):

    opt = SGD(f.parameters(), lr=0.01)
    L = nn.CrossEntropyLoss()

    losses = []
    epochs = []

    for epoch in range(n_epochs):
        print(f'Epoch {epoch}')

        N = len(dl)

        for i, (x, y) in enumerate(dl):

            x = x.to(device)
            y = y.to(device)

            opt.zero_grad()

            loss_value = L(f(x), y)

            loss_value.backward()

            opt.step()

            epochs.append(epoch + i/N)
            losses.append(loss_value.item())

    return np.array(epochs), np.array(losses)

In [21]:
train_dl = DataLoader(train, batch_size=512, shuffle=True)
test_dl = DataLoader(test, batch_size=512, shuffle=False)
epoch_data, loss_data = train_model(train_dl, f)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19


In [22]:
def compute_accuracy(dl, model, device):

    model.eval()  
    correct = 0
    total = 0

    with torch.no_grad():  
        for x, y in dl:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = outputs.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [24]:
train_acc = compute_accuracy(train_dl, f, device)
test_acc = compute_accuracy(test_dl, f, device)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

Train accuracy: 0.8826333333333334
Test accuracy: 0.8841
